<a href="https://colab.research.google.com/github/kader-xai/data-science-roadmap/blob/main/module_44_vllm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 🚀 **Module 44 — vLLM (high-throughput LLM inference for production)** &nbsp;·&nbsp; part of [`data-science-roadmap`](https://github.com/kader-xai/data-science-roadmap)


# Module 44 — vLLM

> Ollama (M38) is great for laptops and a single user. **vLLM** is what you run when you have a fleet of GPUs and thousands of concurrent users. It's the **production inference server** that powers most "OpenAI-compatible" deployments — Together AI, Anyscale, Fireworks, plus thousands of self-hosted setups. Its core trick is **PagedAttention** (manage the KV-cache like an OS manages memory), which delivers ~24× the throughput of naive HuggingFace generate() at the same latency.

### What you'll cover
1. Why vLLM — PagedAttention + continuous batching in plain English
2. Throughput vs latency — what you're really optimising
3. Setup — install vLLM (GPU runtime required)
4. Offline batched inference — `LLM(...).generate(prompts)`
5. Sampling parameters that actually matter
6. **vLLM serve** — start an OpenAI-compatible HTTP server
7. Hitting it from the OpenAI Python SDK (same code as M38, different port)
8. **Tensor parallelism** — splitting one model across many GPUs
9. **Prefix caching** + **speculative decoding** — the two big wins
10. Picking the right server: vLLM vs TGI vs SGLang vs llama-server (M38)

> 🟡 You need a **GPU runtime** for this notebook. On free Colab T4 you can fit a 0.5–1.5 B model. Bigger models need an A100 / H100 paid tier, or a real cluster.


## 1 · Why vLLM — PagedAttention + continuous batching

A naive HuggingFace `model.generate(...)` has two problems at scale:

**Problem 1: KV-cache fragmentation.** Each request reserves a contiguous block big enough for its **maximum** possible output (say 2048 tokens). Most requests finish at 200 tokens → 90% of the GPU's KV memory is wasted on holes.

→ **PagedAttention.** vLLM treats KV memory like an OS treats RAM: split it into fixed-size **pages** (typically 16 tokens), allocate pages on-demand as the sequence grows. No more fragmentation, ~3-5× more concurrent requests fit.

**Problem 2: Static batching.** A naive batched generation waits for the slowest request in the batch to finish before starting the next batch. If one request needs 500 tokens and 31 others need 50, the GPU sits idle for the long tail.

→ **Continuous batching.** vLLM swaps finished requests *out* of the batch and new ones *in* every iteration. The GPU is always full. ~10× higher throughput at the same latency.

These two together — plus optimised CUDA kernels — are why vLLM dominates production.

## 2 · Throughput vs latency

| You care about | What to tune |
|---|---|
| **TTFT** — time to first token (chat UI feel) | small batch, high `gpu_memory_utilization`, **prefix cache** for system prompts |
| **TPOT** — time per output token (streaming pace) | enable **speculative decoding**, ensure tensor-parallel covers attention |
| **Throughput** — total tokens/sec across all users | big `max_num_seqs`, enable **chunked prefill**, batch many requests |
| **Cost / request** | smaller model, AWQ/GPTQ quant, `--enforce-eager=False`, multi-LoRA on one base |

Most teams start by maximising throughput (cheap), then trade some throughput for latency once the bill is OK.

## 3 · Setup

In [ ]:
!nvidia-smi | head -8

In [ ]:
# vLLM 0.6+ pins specific torch / xformers builds — let pip resolve
!pip -q install vllm==0.6.4

## 4 · Offline batched inference

In [ ]:
from vllm import LLM, SamplingParams

# Tiny model so it fits on the free T4 (16 GB)
llm = LLM(model="Qwen/Qwen2.5-0.5B-Instruct",
          gpu_memory_utilization=0.5,   # keep some VRAM free for tooling
          max_model_len=2048)

prompts = [
    "What is RAG in one sentence?",
    "List 3 reasons to use a vector DB.",
    "Explain HNSW like I'm 12.",
    "What is PagedAttention?",
]

params = SamplingParams(temperature=0.0, max_tokens=128)
outputs = llm.generate(prompts, params)

for o in outputs:
    print("Q:", o.prompt[:60], "...")
    print("A:", o.outputs[0].text.strip(), "\n")

**Notice the API.** `llm.generate(list_of_prompts)` returns one result per prompt. vLLM **batches them automatically**, packs prefill + decode work into one engine step (continuous batching), and runs ~10× faster than calling HF `generate()` in a Python loop.

## 5 · Sampling parameters that matter

In [ ]:
params = SamplingParams(
    temperature=0.7,        # 0 = greedy; >1 = more random
    top_p=0.9,              # nucleus sampling: keep tokens whose cumulative prob ≤ 0.9
    top_k=-1,               # -1 = disabled
    repetition_penalty=1.05,
    max_tokens=256,
    stop=["\n\n", "<|endoftext|>"],
    seed=42,                # reproducible sampling
)
out = llm.generate(["Write a haiku about CUDA cores."], params)
print(out[0].outputs[0].text)

For tool-use / structured outputs vLLM also supports **guided decoding** with grammars / JSON schema:

```python
from vllm import SamplingParams
from vllm.sampling_params import GuidedDecodingParams

guided = GuidedDecodingParams(json={"type":"object", "properties":{"name":{"type":"string"},"age":{"type":"integer"}}, "required":["name","age"]})
params = SamplingParams(temperature=0.0, max_tokens=128, guided_decoding=guided)
```


## 6 · vLLM serve — OpenAI-compatible HTTP

In [ ]:
# Spawn vLLM's OpenAI-compatible server in the background
import subprocess, time
srv = subprocess.Popen([
    "python","-m","vllm.entrypoints.openai.api_server",
    "--model", "Qwen/Qwen2.5-0.5B-Instruct",
    "--gpu-memory-utilization", "0.5",
    "--max-model-len", "2048",
    "--port", "8000",
], stdout=open("/tmp/vllm.log","w"), stderr=subprocess.STDOUT)

# wait for health
for _ in range(60):
    time.sleep(2)
    r = subprocess.run(["curl","-s","-o","/dev/null","-w","%{http_code}",
                        "http://localhost:8000/v1/models"], capture_output=True, text=True)
    if r.stdout == "200":
        print("ready"); break
else:
    print("timeout — check /tmp/vllm.log")

## 7 · Hitting it like OpenAI

In [ ]:
!pip -q install openai
from openai import OpenAI

client = OpenAI(base_url="http://localhost:8000/v1", api_key="x")

resp = client.chat.completions.create(
    model="Qwen/Qwen2.5-0.5B-Instruct",
    messages=[
        {"role":"system","content":"You are a terse assistant."},
        {"role":"user","content":"What's PagedAttention?"},
    ],
)
print(resp.choices[0].message.content)

In [ ]:
# Streaming works the same as Ollama / OpenAI — same client code
stream = client.chat.completions.create(
    model="Qwen/Qwen2.5-0.5B-Instruct",
    messages=[{"role":"user","content":"List 3 things vLLM optimises."}],
    stream=True,
)
for chunk in stream:
    d = chunk.choices[0].delta.content
    if d: print(d, end="", flush=True)
print()

**The point.** Your application code from M38 (Ollama) and from the OpenAI cloud doesn't change at all when you swap the backend to vLLM — only `base_url` does. **This is the whole reason to keep your client code OpenAI-shaped.**

## 8 · Tensor parallelism — splitting one model across GPUs

A 70 B model in fp16 needs ~140 GB of VRAM — no single GPU has that. **Tensor parallelism** splits each weight matrix across `N` GPUs; each GPU does a piece of the matmul, then the results are all-reduced.

```bash
# 8× A100 serving Llama-3-70B
python -m vllm.entrypoints.openai.api_server \
    --model meta-llama/Meta-Llama-3-70B-Instruct \
    --tensor-parallel-size 8 \
    --max-model-len 8192
```

Other parallelism axes:
- **Pipeline parallelism** — split *layers* across GPUs (longer pipeline, less comms; rarely needed below 100B).
- **Data parallelism** — replicate the whole model across GPUs and serve different requests on each. Linear scale-out.

**Rule of thumb.** If the model fits on one GPU → use **data parallelism** (more replicas). If it doesn't → **tensor parallelism**, then add data-parallel replicas.

## 9 · Prefix caching + speculative decoding

Two features you turn on in production and forget about:

### Prefix caching (`--enable-prefix-caching`)
If many requests share a prefix (system prompt, RAG context, function-call schema), vLLM **caches the KV-state** of that prefix and reuses it across requests. Saves the entire prefill cost on cache hits — often 70-90% of total compute for chat workloads.

### Speculative decoding (`--speculative-model`)
Run a tiny **draft model** (e.g. 1B) to propose `k` tokens; the big model verifies them in **one** forward pass. ~1.5–3× decode speedup with no quality loss. Conceptually it's "two cheap calls vs one expensive call" — the big model agrees with the draft most of the time.

```bash
python -m vllm.entrypoints.openai.api_server \
    --model Qwen/Qwen2.5-72B-Instruct \
    --speculative-model Qwen/Qwen2.5-0.5B-Instruct \
    --num-speculative-tokens 5 \
    --enable-prefix-caching
```


## 10 · Picking the right server

| Server | Best at | Watch out for |
|---|---|---|
| **vLLM** | best general-purpose throughput; PagedAttention; multi-LoRA | bleeding-edge model support sometimes lags by days |
| **TGI** (Text Generation Inference, HF) | mature; tight HF integration; tool calling | harder to extend; license restrictions on newer versions |
| **SGLang** | structured generation; fastest for **agentic** workloads with re-prefill | smaller ecosystem |
| **TensorRT-LLM** | absolute fastest on NVIDIA; ahead-of-time compile | brittle; long compile; NVIDIA-only |
| **llama-server** (M38) | local laptops, edge, no-GPU CPUs | per-request batching only |
| **Ollama** (M38) | dev convenience | not production at scale |

**Default pick in 2026.** vLLM for fleets of NVIDIA GPUs. SGLang if your workload is heavy on tool-calling / structured outputs. TensorRT-LLM only if you've hit vLLM's ceiling and have an NVIDIA-only deploy.

### Costs you should track
- **$/Mtok input** and **$/Mtok output** — your unit economics
- **GPU utilisation** (`nvidia-smi`, DCGM) — should be ≥ 80% under load
- **KV-cache hit rate** — if low, prefix caching isn't helping → revisit your system prompt


## ✅ Recap
- **PagedAttention** + **continuous batching** = vLLM's two superpowers.
- One line: `LLM(model=...).generate(prompts)` for offline batches.
- `python -m vllm.entrypoints.openai.api_server` for an OpenAI-compatible HTTP server.
- **Tensor parallel** to fit big models; **prefix caching** + **speculative decoding** for production speed.
- Keep client code OpenAI-shaped → swap Ollama / vLLM / OpenAI by changing `base_url`.

Cleanup:


In [ ]:
try: srv.terminate()
except: pass

Next: **M45 — gRPC** (service-to-service plumbing for AI backends).